Proxy pressure is `abs(delta_weight) / ADV20` until `estimated_flow` exists. The proxy is not written into `estimated_flow`. Quintiles are formed inside each reconstitution and action. Q1 is the lowest score.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from rebalance.calendar import TradingCalendar
from rebalance.config import load_research_config, repo_root, resolve_config_path
from rebalance.data.prices import load_fundamentals, load_prices
from rebalance.data.russell import load_announcements
from rebalance.data.securities import load_security_master
from rebalance.events.revisions import event_state_asof
from rebalance.events.timeline import select_tradable_events

root = repo_root()
config = load_research_config(root / "config" / "research.yaml")
calendar = TradingCalendar(config.calendar_name, config.timezone)
master = load_security_master(resolve_config_path(root, config.paths.security_master))
events = load_announcements(
    resolve_config_path(root, config.paths.announcements),
    master,
    timezone=config.timezone,
)
trades, excluded = select_tradable_events(events, calendar)
print("Synthetic fixtures only. Do not read these prints as a Russell result.")
print(f"announcements={len(events)} tradable={len(trades)} excluded={len(excluded)}")


Synthetic fixtures only. Do not read these prints as a Russell result.
announcements=23 tradable=20 excluded=1


In [2]:
from rebalance.features.liquidity import compute_liquidity_features
from rebalance.features.pressure import compute_pressure
from rebalance.research.cross_section import assign_pressure_quintiles, quintile_car_table
from rebalance.research.event_study import build_event_panel, window_cars

prices = load_prices(resolve_config_path(root, config.paths.prices), calendar=calendar)
fundamentals = load_fundamentals(resolve_config_path(root, config.paths.fundamentals))
features, feature_diagnostics = compute_liquidity_features(
    trades, prices, calendar, config.benchmark_security_id, fundamentals
)
pressure = compute_pressure(features)
print(pressure["pressure_kind"].value_counts().to_string())
print("non-null estimated_flow rows:", int(pressure["estimated_flow"].notna().sum()))
print("proxy rows:", int(pressure["pressure_is_proxy"].sum()), "feature diagnostics:", len(feature_diagnostics))

panel, _ = build_event_panel(trades, prices, calendar, config.benchmark_security_id)
cars = window_cars(panel, calendar, config.windows)
primary = cars[cars["window"] == config.primary_window].merge(
    pressure[["rebalance_id", "security_id", "pressure_score", "pressure_kind", "pressure_is_proxy"]],
    on=["rebalance_id", "security_id"],
    how="left",
)
table = quintile_car_table(assign_pressure_quintiles(primary))
print("Synthetic quintile CAR. Not a test of index-demand monotonicity.")
display(table)


pressure_kind
abs_delta_weight_over_adv20    20
non-null estimated_flow rows: 0
proxy rows: 20 feature diagnostics: 0


Synthetic quintile CAR. Not a test of index-demand monotonicity.


,action,pressure_quintile,n_names,n_rebalances,mean_car_event_equal,median_car_pooled,mean_car_pooled
0,ADD,Q1,2,2,-0.012707,-0.012707,-0.012707
1,ADD,Q2,2,2,0.015552,0.015552,0.015552
2,ADD,Q3,2,2,0.051108,0.051108,0.051108
3,ADD,Q4,2,2,0.054682,0.054682,0.054682
4,ADD,Q5,2,2,0.006313,0.006313,0.006313
5,DELETE,Q1,2,2,0.011215,0.011215,0.011215
6,DELETE,Q2,2,2,-0.024923,-0.024923,-0.024923
7,DELETE,Q3,2,2,0.035570,0.035570,0.035570
8,DELETE,Q4,2,2,0.012990,0.012990,0.012990
9,DELETE,Q5,2,2,-0.028686,-0.028686,-0.028686
